# XJTU discharge PPO pilot

Train a small PPO agent and compare it with a fixed schedule, a simple rule, and a constrained lookahead controller on held-out Random Walk batteries. Every action selects a schedule for the first constant-current plateau of each of ten operating cycles. Schedules originate in the training logs. Later pulses and cutoff behavior remain part of the observed Random Walk regime.

This is a simulated three-reference-test pilot. The notebook reads the Notebook 19 report folder or ZIP. It uses one CPU thread. The project-local Gymnasium installation can be reproduced with `uv pip install --target .venv/notebook20-deps --no-deps gymnasium==1.2.2 farama-notifications==0.0.4`. The existing environment also supplies NumPy, pandas, scikit-learn, cloudpickle, PyTorch, and matplotlib.

In [1]:
import os
for thread_var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
                   "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[thread_var] = "1"

import json
import sys
from itertools import product
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from torch import nn
from torch.distributions import Categorical
from threadpoolctl import threadpool_limits

torch.set_num_threads(1)

cwd = Path.cwd()
project = next((p for p in (cwd, *cwd.parents) if (p / "README_LifeGuard_RL.md").is_file()), None)
if project is None:
    raise FileNotFoundError("Open this notebook from within the LifeGuard-RL project.")
local_dependencies = project / ".venv" / "notebook20-deps"
if local_dependencies.is_dir() and str(local_dependencies) not in sys.path:
    sys.path.insert(0, str(local_dependencies))
import gymnasium as gym

SOURCE_OVERRIDE = None
OUTPUT_ROOT = None
INPUT_NAME = "lifeguard_xjtu_discharge_soh_results"
OUTPUT_NAME = "lifeguard_xjtu_discharge_ppo_results"
source = Path(SOURCE_OVERRIDE).expanduser() if SOURCE_OVERRIDE else project / "reports" / INPUT_NAME
if SOURCE_OVERRIDE is None and not source.is_dir():
    source = source.with_suffix(".zip")
output = (Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT else project / "reports") / OUTPUT_NAME

SEED = 42
PPO_SEEDS = (42, 73, 101)
BOOTSTRAP_DRAWS = 2000
HORIZON = 3
DISTURBANCES = 8
MIN_SOH = 80.0
SOH_RADIUS = 6.0
AGE_RADIUS = 70.0
MIN_CELLS = 2
ENERGY_FRACTION = .90
HIT_TOLERANCE = .05
ENERGY_ALPHA = 10
PPO_UPDATES = 8
SAMPLES_PER_UPDATE = 96
PRESETS = (0, 1, 2)
PROFILE_COLUMNS = ["first_current_mean_a", "initial_current_square_mean_a2",
                   "initial_duration_mean_min", "initial_current_square_minutes"]
POLICIES = {"fixed": "Fixed middle template", "rule": "SOH rule",
            "lookahead": "Constrained lookahead", "ppo": "PPO"}


def read_table(name):
    if source.is_dir():
        return pd.read_csv(source / (name + ".csv"))
    with ZipFile(source) as archive:
        entries = [path for path in archive.namelist() if PurePosixPath(path).name == name + ".csv"]
        if len(entries) != 1:
            raise ValueError(f"Expected one {name}.csv in {source}.")
        with archive.open(entries[0]) as stream:
            return pd.read_csv(stream)

In [2]:
def add_histories(intervals):
    data = intervals.loc[intervals.status.eq("valid")].copy().reset_index(drop=True)
    end_at = {(row.cell_id, row.end_cycle): row for row in data.itertuples(index=False)}
    histories = []
    for row in data.itertuples(index=False):
        previous = end_at.get((row.cell_id, row.start_cycle))
        earlier = end_at.get((row.cell_id, previous.start_cycle)) if previous is not None else None
        histories.append((previous.soh_before if previous is not None else np.nan,
                          earlier.soh_before if earlier is not None else np.nan))
    data[["previous_soh", "earlier_soh"]] = histories
    return data


def chain_indices(data, horizon):
    by_start = {(row.cell_id, row.start_cycle): row.Index for row in data.itertuples()}
    routes = []
    for row in data.itertuples():
        path = [row.Index]
        for _ in range(1, horizon):
            forward = by_start.get((row.cell_id, data.at[path[-1], "end_cycle"]))
            if forward is None:
                break
            path.append(forward)
        if len(path) == horizon:
            routes.append(path)
    return np.asarray(routes, dtype=int).reshape(-1, horizon)


gate = read_table("ppo_gate")
if len(gate) != 1 or not bool(gate.iloc[0].advance_to_ppo):
    raise ValueError("The Notebook 19 report must pass the PPO pilot gate.")
intervals = read_table("reference_intervals")
data = add_histories(intervals.loc[intervals.cohort.eq("RW")])
known = read_table("held_out_predictions")
known = known.loc[known.cohort.eq("RW")]
choices = read_table("model_selection")
choices = choices.loc[choices.cohort.eq("RW") & choices.model.eq("action") & choices.selected]
alpha_by_cell = choices.set_index("cell_id", verify_integrity=True).alpha.to_dict()
cells = sorted(data.cell_id.unique())
cell_ids = data.cell_id.to_numpy()
profiles = data[PROFILE_COLUMNS].to_numpy()
initial = data[["soh_before", "previous_soh", "earlier_soh"]].to_numpy()
ages = data.operating_age.to_numpy()
truth = np.column_stack((data.soh_after - data.soh_before, data.realized_wh))
chains = {h: chain_indices(data, h) for h in (1, HORIZON)}
assert set(alpha_by_cell) == set(cells) and np.isfinite(profiles).all() and np.isfinite(truth).all()


def design(states, age, commands):
    soh, previous, earlier = states.T
    has_prev = np.isfinite(previous)
    recent = np.where(has_prev, soh - previous, 0)
    drift = np.where(np.isfinite(earlier), (soh - earlier) / 2, recent)
    return np.column_stack((soh, age, recent, drift, has_prev, commands))


def fit(train, health_alpha):
    x = design(initial[train], ages[train], profiles[train])
    subset = data.loc[train]
    weights = 1 / subset.groupby("cell_id").start_cycle.transform("size").to_numpy(dtype=float)
    weights /= weights.mean()
    usable = np.ptp(x, axis=0) > 0
    matrix = x[:, usable]
    center = np.average(matrix, axis=0, weights=weights)
    scale = np.sqrt(np.average((matrix - center)**2, axis=0, weights=weights))
    standardized = (matrix - center) / scale
    slopes = np.zeros((x.shape[1], 2))
    intercept = np.empty(2)
    for j, alpha in enumerate((health_alpha, ENERGY_ALPHA)):
        ridge = Ridge(alpha=alpha).fit(standardized, truth[train, j], sample_weight=weights)
        slopes[usable, j] = ridge.coef_ / scale
        intercept[j] = ridge.intercept_ - center @ slopes[usable, j]
    return slopes, intercept


def transition(model, state, age, commands):
    coefficients, intercept = model
    return design(state, age, commands) @ coefficients + intercept


def calibrate(train, held_out):
    errors = np.full_like(truth, np.nan)
    for inner in cells:
        if inner == held_out:
            continue
        score = cell_ids == inner
        fit_rows = train & ~score
        model = fit(fit_rows, health_alpha=10)
        errors[score] = truth[score] - transition(model, initial[score], ages[score], profiles[score])
    paths = chains[HORIZON]
    paths = paths[train[paths[:, 0]]]
    noise = errors[paths]
    if not np.isfinite(noise).all():
        raise ValueError("Incomplete training-only calibration windows.")
    return noise


def measure_forecasts(model, held_out, horizons):
    records = []
    for h, all_paths in horizons.items():
        paths = all_paths[cell_ids[all_paths[:, 0]] == held_out]
        state = initial[paths[:, 0]].copy()
        wh = np.zeros(len(paths))
        for step in paths.T:
            estimate = transition(model, state, ages[step], profiles[step])
            wh += estimate[:, 1]
            state = np.column_stack((state[:, 0] + estimate[:, 0], state[:, :2]))
        for path, predicted, delivered in zip(paths, state[:, 0], wh):
            start = data.iloc[path[0]]
            end = data.iloc[path[-1]]
            records.append((held_out, int(start.start_cycle), int(end.end_cycle), h,
                            predicted, end.soh_after, delivered, float(truth[path, 1].sum())))
    return records


with threadpool_limits(limits=1):
    models, disturbances, forecast_records = {}, {}, []
    for cell in cells:
        train = cell_ids != cell
        models[cell] = fit(train, alpha_by_cell[cell])
        disturbances[cell] = calibrate(train, cell)
        forecast_records.extend(measure_forecasts(models[cell], cell, chains))

forecast_checks = pd.DataFrame(forecast_records, columns=["cell_id", "start_cycle", "end_cycle", "horizon_tests",
    "soh_prediction", "soh_measured", "wh_prediction", "wh_measured"])
joined = forecast_checks.merge(known[["cell_id", "start_cycle", "end_cycle", "horizon_tests", "action"]],
    on=["cell_id", "start_cycle", "end_cycle", "horizon_tests"], validate="one_to_one")
assert len(joined) == len(known) == len(forecast_checks)
np.testing.assert_allclose(joined.soh_prediction, joined.action, atol=1e-9, rtol=0)
print("Reproduced all Notebook 19 held-out action-aware SOH forecasts.")

Reproduced all Notebook 19 held-out action-aware SOH forecasts.


## Supported schedules and operating rules

Each episode has three candidate ten-cycle schedules from training batteries at comparable starting SOH and operating age. They represent low, middle, and high planned squared-current exposure. Each candidate has nearby observations from at least two training batteries in its exposure group. A candidate is excluded at decision time when its group or its source context falls outside the same support rule.

Each control decision covers one RPT interval of ten operating cycles. Each episode contains three intervals, or thirty operating cycles. The energy budget is 90% of the fixed middle template's forecast delivered Wh. Before applying a decision, the shared controller guard checks action support and the forecast budget, reserving the fixed schedule's forecast for remaining steps. If the budget appears infeasible, the guard selects the supported option with the greatest predicted Wh. Actual final shortfall remains part of scoring. All methods face the same guard. This is a statistical operating-domain check in simulation.

Residuals sampled from contiguous inner held-out training paths add joint Wh and SOH variability. Test policies face identical residual draws for each start. Episodes and templates from the test battery are excluded from fitting, calibration, and PPO training.

In [3]:
class ScheduleStudy:
    def __init__(self, held_out, model, noise):
        self.held_out = held_out
        self.model = model
        self.noise = noise
        self.train = np.flatnonzero(cell_ids != held_out)
        self.group_edges = np.quantile(profiles[self.train, -1], [1 / 3, 2 / 3])
        self.groups = np.searchsorted(self.group_edges, profiles[:, -1], side="right")
        self.train_cells = pd.factorize(cell_ids)[0]

    def nearby(self, soh, age, indexes):
        return (np.abs(initial[indexes, 0] - soh) <= SOH_RADIUS) & (np.abs(ages[indexes] - age) <= AGE_RADIUS)

    def supported(self, state, age, templates):
        soh = state[0]
        if soh < MIN_SOH:
            return np.zeros(len(PRESETS), dtype=bool)
        nearby = self.train[self.nearby(soh, age, self.train)]
        counts = [len(set(self.train_cells[nearby[self.groups[nearby] == action]])) for action in PRESETS]
        return self.nearby(soh, age, templates) & (np.asarray(counts) >= MIN_CELLS)

    def predict(self, state, age, template_ids):
        tiled = np.repeat(np.asarray(state)[None, :], len(template_ids), axis=0)
        return transition(self.model, tiled, np.full(len(template_ids), age), profiles[template_ids])

    def build_case(self, origin_index):
        state, age = initial[origin_index].copy(), int(ages[origin_index])
        if state[0] < MIN_SOH or not np.isfinite(state[1]):
            return None
        nearby = self.train[self.nearby(state[0], age, self.train)]
        templates = []
        for action in PRESETS:
            rows = nearby[self.groups[nearby] == action]
            if len(set(self.train_cells[rows])) < MIN_CELLS:
                return None
            distance = np.abs(initial[rows, 0] - state[0]) / SOH_RADIUS + np.abs(ages[rows] - age) / AGE_RADIUS
            templates.append(rows[distance.argmin()])
        templates = np.asarray(templates)
        middle = []
        forecast_state = state.copy()
        for step in range(HORIZON):
            if not self.supported(forecast_state, age + 10 * step, templates)[1]:
                return None
            change, wh = self.predict(forecast_state, age + 10 * step, templates[1:2])[0]
            middle.append(float(wh))
            forecast_state = np.r_[forecast_state[0] + change, forecast_state[:2]]
        if min(middle) <= 0:
            return None
        return {"index": origin_index, "state": state, "age": age, "templates": templates,
                "reserve": np.asarray(middle), "budget": ENERGY_FRACTION * sum(middle)}

    def select_mask(self, state, age, case, step, credited):
        allowed = self.supported(state, age, case["templates"])
        estimates = self.predict(state, age, case["templates"])
        feasible = allowed & (credited + estimates[:, 1] + case["reserve"][step + 1:].sum() >= case["budget"])
        if allowed.any() and not feasible.any():
            recovery = int(np.argmax(np.where(allowed, estimates[:, 1], -np.inf)))
            feasible[recovery] = True
            return feasible, estimates, True
        return feasible, estimates, False


class DischargeEnv(gym.Env):
    def __init__(self, study, cases):
        super().__init__()
        self.study, self.cases = study, cases
        self.action_space = gym.spaces.Discrete(len(PRESETS))
        self.observation_space = gym.spaces.Box(-np.inf, np.inf, shape=(12,), dtype=np.float32)

    def observation(self):
        soh = self.state[0]
        remaining = self.case["budget"] - self.credited
        reserve = self.case["reserve"][self.turn:].sum()
        drift = soh - self.state[1]
        obs = np.r_[(soh - 90) / 10, drift, (self.age - 80) / 100, self.turn / HORIZON,
                    (remaining - reserve) / 10, self.case["budget"] / 60,
                    self.estimates[:, 0], (self.estimates[:, 1] - 55) / 15]
        return obs.astype(np.float32)

    def update_mask(self):
        self.mask, self.estimates, self.recovery = self.study.select_mask(
            self.state, self.age, self.case, self.turn, self.credited)

    def info(self):
        return dict(action_mask=self.mask.copy(), soh=float(self.state[0]), wh=float(self.credited),
                    budget=self.case["budget"], guard_recoveries=self.guard_recoveries,
                    unsupported_stop=self.unsupported_stop)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        options = options or {}
        self.case = options.get("case") or self.cases[int(self.np_random.integers(len(self.cases)))]
        if "noise" in options:
            self.noise = options["noise"]
        else:
            self.noise = self.study.noise[int(self.np_random.integers(len(self.study.noise)))]
        self.state, self.age = self.case["state"].copy(), self.case["age"]
        self.turn, self.credited, self.guard_recoveries, self.unsupported_stop = 0, 0.0, 0, False
        self.update_mask()
        assert self.mask.any()
        return self.observation(), self.info()

    def step(self, action):
        if not self.mask[action]:
            raise ValueError("The requested action is masked by the shared controller guard.")
        if self.recovery:
            self.guard_recoveries += 1
        change, wh = self.estimates[action] + self.noise[self.turn]
        delivered = max(0.0, float(wh))
        self.state = np.r_[self.state[0] + change, self.state[:2]]
        self.turn += 1
        self.age += 10
        self.credited += delivered
        finished = self.turn == HORIZON
        if not finished:
            self.update_mask()
            self.unsupported_stop = not self.mask.any()
        else:
            self.mask = np.zeros(len(PRESETS), dtype=bool)
        done = finished or self.unsupported_stop
        reward = float(change) - (3.0 * max(0, self.case["budget"] - self.credited) if done else 0)
        if self.unsupported_stop:
            reward -= 5
        return self.observation(), reward, done, False, self.info()


def all_cases(study):
    starts = chains[HORIZON][:, 0]
    training, testing, rejected = [], [], []
    for index in starts:
        case = study.build_case(index)
        test = cell_ids[index] == study.held_out
        if case is not None:
            (testing if test else training).append(case)
        elif test:
            rejected.append(index)
    if not training:
        raise ValueError(f"No supported training starts for {study.held_out}.")
    return training, testing, rejected

## PPO and held-out comparison

The actor and critic share a small CPU network. PPO uses a clipped likelihood-ratio objective with masked discrete actions, bootstrapped advantages, and three fixed seeds per held-out battery. Counterpart policies use the same masks and energy budget. The fixed baseline keeps the middle observed template. The rule prefers low exposure below 90% SOH and the middle template otherwise. The lookahead controller enumerates the remaining supported schedules and replans after every interval.

Retain PPO only if the paired 95% interval supports a lower modeled SOH loss than lookahead, the budget hit rate is within five percentage points of lookahead, and PPO completes its episodes within support. Positive SOH changes remain in the modeled outcome.

In [4]:
class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(12, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh())
        self.actor, self.critic = nn.Linear(32, 3), nn.Linear(32, 1)

    def forward(self, observations, mask):
        hidden = self.trunk(observations)
        logits = self.actor(hidden).masked_fill(~mask, -1e9)
        return Categorical(logits=logits), self.critic(hidden).squeeze(-1)


def train_ppo(env, seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    actor = PolicyNet()
    optimizer = torch.optim.Adam(actor.parameters(), lr=0.001)
    observation, info = env.reset(seed=seed)
    log = []
    for update in range(PPO_UPDATES):
        examples = []
        for _ in range(SAMPLES_PER_UPDATE):
            mask = info["action_mask"]
            with torch.no_grad():
                distribution, value = actor(torch.as_tensor(observation), torch.as_tensor(mask))
                action = distribution.sample()
                likelihood = distribution.log_prob(action)
            current = observation
            observation, reward, done, _, info = env.step(int(action))
            examples.append((current, mask, int(action), float(likelihood), float(value), reward, done))
            if done:
                observation, info = env.reset()
        with torch.no_grad():
            _, value = actor(torch.as_tensor(observation), torch.as_tensor(info["action_mask"]))
        bootstrap = float(value)

        observations, masks, actions, likelihood, values, rewards, done = map(np.asarray, zip(*examples))
        advantages = np.zeros(len(rewards), dtype=np.float32)
        next_value, effect = bootstrap, 0.0
        for i in reversed(range(len(rewards))):
            alive = 1 - int(done[i])
            delta = rewards[i] + alive * next_value - values[i]
            effect = delta + .95 * alive * effect
            advantages[i] = effect
            next_value = values[i]
        returns = advantages + values.astype(np.float32)
        standardized = (advantages - advantages.mean()) / max(float(advantages.std()), 1e-8)

        obs = torch.as_tensor(observations)
        masks = torch.as_tensor(masks)
        acts = torch.as_tensor(actions)
        old = torch.as_tensor(likelihood, dtype=torch.float32)
        gains = torch.as_tensor(standardized)
        returns = torch.as_tensor(returns)
        for _ in range(4):
            order = rng.permutation(len(rewards)).reshape(-1, 24)
            for batch in order:
                distribution, values_new = actor(obs[batch], masks[batch])
                ratio = (distribution.log_prob(acts[batch]) - old[batch]).exp()
                clipped = torch.clamp(ratio, .8, 1.2)
                actor_loss = -torch.minimum(ratio * gains[batch], clipped * gains[batch]).mean()
                critic_loss = (values_new - returns[batch]).square().mean()
                loss = actor_loss + .5 * critic_loss - .01 * distribution.entropy().mean()
                if not torch.isfinite(loss):
                    raise ValueError("PPO produced a nonfinite loss.")
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(actor.parameters(), .5)
                optimizer.step()
        log.append((seed, update + 1, (update + 1) * SAMPLES_PER_UPDATE, float(np.mean(rewards)), float(loss.detach())))
    return actor.eval(), log


def choose_lookahead(env):
    study, case = env.study, env.case

    def utility(state, age, step, credited):
        if step == HORIZON:
            return float(state[0] - 3 * max(0, case["budget"] - credited))
        mask, estimates, _ = study.select_mask(state, age, case, step, credited)
        if not mask.any():
            return float(state[0] - 5 - 3 * max(0, case["budget"] - credited))
        return max(follow(state, age, step, credited, estimates[action]) for action in np.flatnonzero(mask))

    def follow(state, age, step, credited, estimate):
        change, wh = estimate
        next_state = np.r_[state[0] + change, state[:2]]
        return utility(next_state, age + 10, step + 1, credited + max(0, wh))

    return min(np.flatnonzero(env.mask), key=lambda action:
               (-follow(env.state, env.age, env.turn, env.credited, env.estimates[action]), int(action)))


def baseline_action(env, policy):
    if policy == "lookahead":
        return choose_lookahead(env)
    preferred = 0 if policy == "rule" and env.state[0] < 90 else 1
    if env.mask[preferred]:
        return preferred
    return int(np.argmax(np.where(env.mask, env.estimates[:, 1], -np.inf)))


def evaluate_episode(env, case, noise, policy, actor=None):
    observation, info = env.reset(options={"case": case, "noise": noise})
    total_steps = 0
    actions = []
    while True:
        if policy == "ppo":
            with torch.no_grad():
                distribution, _ = actor(torch.as_tensor(observation), torch.as_tensor(info["action_mask"]))
                action = int(distribution.logits.argmax())
        else:
            action = baseline_action(env, policy)
        actions.append(int(action))
        observation, _, done, _, info = env.step(action)
        total_steps += 1
        if done:
            break
    meets_budget = info["wh"] >= info["budget"] and not info["unsupported_stop"]
    return dict(soh_loss_pp=case["state"][0] - info["soh"], delivered_wh=info["wh"], budget_wh=info["budget"],
                budget_hit=meets_budget, guard_recoveries=info["guard_recoveries"],
                unsupported_stop=info["unsupported_stop"], intervals_completed=total_steps,
                actions="".join(map(str, actions)))


def evaluate_folds():
    episodes, logs, coverage, catalog = [], [], [], []
    checkpoints = {}
    rng = np.random.default_rng(SEED)
    for held_out in cells:
        study = ScheduleStudy(held_out, models[held_out], disturbances[held_out])
        training, testing, rejected = all_cases(study)
        assert all(cell_ids[case["index"]] != held_out for case in training)
        assert all(cell_ids[case["index"]] == held_out for case in testing)
        assert all(cell_ids[index] != held_out for case in [*training, *testing] for index in case["templates"])
        coverage.append((held_out, len(training), len(testing), len(rejected)))
        if not testing:
            continue
        env = DischargeEnv(study, training)
        actors = {}
        for seed in PPO_SEEDS:
            actor, log = train_ppo(env, seed)
            actors[seed] = actor
            checkpoints[f"{held_out}/seed-{seed}"] = actor.state_dict()
            logs.extend((held_out, *row) for row in log)
        for case in testing:
            start = data.iloc[case["index"]]
            for action, index in enumerate(case["templates"]):
                row = data.iloc[index]
                catalog.append((held_out, int(start.start_cycle), action, row.cell_id,
                                int(row.start_cycle), float(row.initial_current_square_minutes)))
            draws = study.noise[rng.integers(len(study.noise), size=DISTURBANCES)]
            for draw, noise in enumerate(draws):
                for policy in POLICIES:
                    agents = actors.items() if policy == "ppo" else [(0, None)]
                    for seed, actor in agents:
                        info = evaluate_episode(env, case, noise, policy, actor)
                        episodes.append(dict(cell_id=held_out, start_cycle=int(start.start_cycle), draw=draw,
                                             policy=policy, seed=seed, **info))
        print(f"Held out {held_out}: {len(testing)} compatible starts, "
              f"{len(rejected)} excluded before simulation, {len(actors)} trained seeds", flush=True)
    return (pd.DataFrame(episodes),
            pd.DataFrame(logs, columns=["cell_id", "seed", "update", "train_steps", "mean_reward", "last_loss"]),
            pd.DataFrame(coverage, columns=["cell_id", "training_starts", "tested_starts", "excluded_starts"]),
            pd.DataFrame(catalog, columns=["cell_id", "start_cycle", "action", "template_battery",
                                          "template_start_cycle", "template_current_square_minutes"]),
            checkpoints)


with threadpool_limits(limits=1):
    episodes, training_log, coverage, templates, checkpoints = evaluate_folds()
if episodes.empty:
    raise ValueError("The held-out evaluation has zero supported starts.")

Held out XJTU_RW_battery-1: 12 compatible starts, 2 excluded before simulation, 3 trained seeds
Held out XJTU_RW_battery-2: 22 compatible starts, 2 excluded before simulation, 3 trained seeds
Held out XJTU_RW_battery-3: 25 compatible starts, 2 excluded before simulation, 3 trained seeds
Held out XJTU_RW_battery-4: 17 compatible starts, 2 excluded before simulation, 3 trained seeds
Held out XJTU_RW_battery-5: 18 compatible starts, 3 excluded before simulation, 3 trained seeds
Held out XJTU_RW_battery-6: 14 compatible starts, 2 excluded before simulation, 3 trained seeds
Held out XJTU_RW_battery-7: 12 compatible starts, 1 excluded before simulation, 3 trained seeds
Held out XJTU_RW_battery-8: 13 compatible starts, 2 excluded before simulation, 3 trained seeds


In [5]:
def paired_interval(values):
    random = np.random.default_rng(SEED)
    indexes = random.integers(len(values), size=(BOOTSTRAP_DRAWS, len(values)))
    return np.quantile(values[indexes].mean(axis=1), [0.025, 0.975])


def metrics_for_episodes(episodes):
    columns = ["soh_loss_pp", "delivered_wh", "budget_hit", "guard_recoveries", "unsupported_stop"]
    by_start = episodes.groupby(["policy", "cell_id", "start_cycle"])[columns].mean()
    by_battery = by_start.groupby(level=["policy", "cell_id"]).mean()
    rows = []
    for policy, group in by_battery.groupby(level="policy"):
        checks = episodes.loc[episodes.policy.eq(policy)]
        rows.append(dict(policy=policy, batteries=len(group), starts=len(by_start.loc[policy]),
                         simulations=len(checks), **group.mean().to_dict(),
                         worst_soh_loss_pp=checks.soh_loss_pp.max(),
                         worst_energy_shortfall_wh=np.maximum(0, checks.budget_wh - checks.delivered_wh).max()))
    metrics = pd.DataFrame(rows).set_index("policy").reindex(POLICIES)
    comparison = by_battery.loc["ppo"] - by_battery.loc["lookahead"]
    loss_low, loss_high = paired_interval(comparison.soh_loss_pp.to_numpy())
    hit_low, hit_high = paired_interval(comparison.budget_hit.to_numpy())
    lower_loss = loss_high < 0
    energy = hit_low >= -HIT_TOLERANCE
    supported = not bool(episodes.loc[episodes.policy.eq("ppo"), "unsupported_stop"].any())
    decision = pd.DataFrame([dict(ppo_minus_lookahead_loss_pp=comparison.soh_loss_pp.mean(),
                                  loss_ci95_low_pp=loss_low, loss_ci95_high_pp=loss_high,
                                  ppo_minus_lookahead_hit_rate=comparison.budget_hit.mean(),
                                  hit_ci95_low=hit_low, hit_ci95_high=hit_high,
                                  better_soh_loss=bool(lower_loss), comparable_energy=bool(energy),
                                  completed_all_in_support=supported,
                                  retain_ppo=bool(lower_loss and energy and supported))])
    return metrics, decision


def score_forecasts(forecasts):
    scores = []
    for count, rows in forecasts.groupby("horizon_tests"):
        for target in ("soh", "wh"):
            error = rows[f"{target}_prediction"] - rows[f"{target}_measured"]
            scores.append(dict(horizon_tests=count, target=target, windows=len(rows),
                               mae=error.abs().groupby(rows.cell_id).mean().mean(),
                               bias=error.groupby(rows.cell_id).mean().mean(), max_error=error.abs().max()))
    return pd.DataFrame(scores)


metrics, decision = metrics_for_episodes(episodes)
forecast_metrics = score_forecasts(forecast_checks)
assert metrics.starts.nunique() == 1 and metrics.batteries.nunique() == 1
assert not templates.cell_id.eq(templates.template_battery).any()
assert int(metrics.loc["ppo", "simulations"]) == int(metrics.loc["fixed", "simulations"]) * len(PPO_SEEDS)
assert coverage.tested_starts.sum() + coverage.excluded_starts.sum() == len(chains[HORIZON])
print(metrics[["starts", "soh_loss_pp", "delivered_wh", "budget_hit", "unsupported_stop"]].round(4).to_string())
print(decision.to_string(index=False))

           starts  soh_loss_pp  delivered_wh  budget_hit  unsupported_stop
policy                                                                    
fixed         133       1.9844      189.7686      0.9961            0.0039
rule          133       1.9880      189.7613      0.9961            0.0039
lookahead     133       1.7547      190.2481      0.9961            0.0039
ppo           133       1.9853      189.7886      0.9961            0.0039
 ppo_minus_lookahead_loss_pp  loss_ci95_low_pp  loss_ci95_high_pp  ppo_minus_lookahead_hit_rate  hit_ci95_low  hit_ci95_high  better_soh_loss  comparable_energy  completed_all_in_support  retain_ppo
                    0.230644          0.202108            0.25734                           0.0           0.0            0.0            False               True                     False       False


In [6]:
def summary(metrics, decision, forecast_metrics, coverage):
    test = decision.iloc[0]
    verdict = ("PPO met the retention criteria in this simulation."
               if test.retain_ppo else "PPO fell short of the retention criteria in this simulation.")
    lines = [
        "# XJTU discharge PPO pilot", "",
        "## Conclusion", "",
        f"**{verdict}** PPO trained on the validated Random Walk SOH transition model, with a separate "
        "delivered-Wh model. The benchmark compares the change in reference-test SOH at a shared energy budget.", "",
        f"Evaluation used {int(coverage.tested_starts.sum())} starts on "
        f"{int((coverage.tested_starts > 0).sum())} held-out batteries. The support checks excluded "
        f"{int(coverage.excluded_starts.sum())} additional starts. Each evaluated start used {DISTURBANCES} shared "
        f"disturbance paths and PPO used {len(PPO_SEEDS)} seeds per battery fold.", "",
        "## Policy comparison over three reference-test intervals", "",
        "| Policy | Mean SOH loss (pp) | Worst SOH loss (pp) | Mean energy (Wh) | Budget hit rate | Support stops |",
        "| --- | ---: | ---: | ---: | ---: | ---: |",
    ]
    for name, row in metrics.iterrows():
        lines.append(f"| {POLICIES[name]} | {row.soh_loss_pp:.3f} | {row.worst_soh_loss_pp:.3f} | "
                     f"{row.delivered_wh:.2f} | {row.budget_hit:.1%} | {row.unsupported_stop:.1%} |")
    lines.extend([
        "",
        "Mean outcomes give each held-out battery equal weight. A positive SOH loss means lower capacity "
        "at the endpoint. A support stop ended a trajectory early and counted as a budget miss.", "",
        f"The PPO-minus-lookahead SOH loss difference was **{test.ppo_minus_lookahead_loss_pp:+.3f} pp** "
        f"(95% interval {test.loss_ci95_low_pp:+.3f} to {test.loss_ci95_high_pp:+.3f}). "
        f"The budget hit rate difference was {test.ppo_minus_lookahead_hit_rate * 100:+.2f} percentage points "
        f"(95% interval {test.hit_ci95_low * 100:+.2f} to {test.hit_ci95_high * 100:+.2f}). "
        "A paired bootstrap resampled batteries with the fitted policies and disturbances held fixed.", "",
        "## Recorded-trajectory forecast check", "",
        "| Tests ahead | Target | Windows | Mean absolute error | Max error |",
        "| --- | --- | ---: | ---: | ---: |",
    ])
    for row in forecast_metrics.itertuples(index=False):
        label = "SOH (percentage points)" if row.target == "soh" else "Delivered energy (Wh)"
        lines.append(f"| {row.horizon_tests} | {label} | {row.windows} | {row.mae:.3f} | {row.max_error:.3f} |")
    lines.extend([
        "",
        "The SOH forecasts reproduce Notebook 19. This check feeds back predicted SOH after the initial state "
        "and uses the recorded first-plateau schedules. The Wh model is fit and evaluated using separate batteries.", "",
        "## Scope", "",
        "Each decision selects a supported ten-cycle template, with one current and duration for the first "
        "discharge plateau of each cycle. The fixed baseline repeats the middle-exposure observed template. "
        "All policies share the operating-domain check and the 90% energy budget. The remaining pulse profile "
        "and delivered energy are represented by the fitted Random Walk transition.", "",
        "The benchmark is a comparison inside the simulator. It covers modeled reference-test SOH over up to "
        "30 operating cycles. Prospective cycling would be needed to establish an effect on battery lifetime.", "",
        "[Original XJTU experimental protocols](https://wang-fujin.github.io/) · "
        "[Gymnasium environment API](https://gymnasium.farama.org/api/env/)", "",
    ])
    return "\n".join(lines)


output.mkdir(parents=True, exist_ok=True)
tables = {"policy_metrics": metrics.reset_index(), "ppo_decision": decision, "coverage": coverage,
          "test_templates": templates, "test_episodes": episodes, "training_log": training_log,
          "recorded_forecasts": forecast_checks, "recorded_forecast_metrics": forecast_metrics}
for name, table in tables.items():
    table.to_csv(output / f"{name}.csv", index=False)
config = dict(action_templates=["low", "middle", "high"], operating_cycles_per_action=10, horizon_tests=HORIZON,
              seeds=PPO_SEEDS, disturbance_paths_per_start=DISTURBANCES, evaluation_seed=SEED,
              updates=PPO_UPDATES, samples_per_update=SAMPLES_PER_UPDATE, budget_fraction=ENERGY_FRACTION,
              energy_alpha=ENERGY_ALPHA, soh_alpha_by_battery=alpha_by_cell, min_soh=MIN_SOH,
              support_soh_radius_pp=SOH_RADIUS, support_age_radius=AGE_RADIUS, support_min_training_cells=MIN_CELLS,
              gymnasium_version=gym.__version__, torch_version=str(torch.__version__))
(output / "run_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
(output / "summary.md").write_text(summary(metrics, decision, forecast_metrics, coverage), encoding="utf-8")
torch.save(checkpoints, output / "ppo_pilot_weights.pt")
files = [*(f"{name}.csv" for name in tables), "run_config.json", "summary.md", "ppo_pilot_weights.pt"]
archive_path = output.with_suffix(".zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as bundle:
    for name in files:
        bundle.write(output / name, arcname=f"{output.name}/{name}")
print(f"Report: {output / 'summary.md'}")
print(f"Download: {archive_path}")

Report: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_xjtu_discharge_ppo_results/summary.md
Download: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_xjtu_discharge_ppo_results.zip
